In [ ]:
# @title Complete HiAGM Pipeline with ModernBERT & Early Stopping
import os
import zipfile
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from google.colab import files
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import collections
from datasets import load_from_disk, load_dataset
from tqdm.auto import tqdm

# --- 1. Setup & Installation ---
print("Installing dependencies...")
!pip install -q transformers datasets accelerate scikit-learn

# --- 2. Configuration ---
CONFIG = {
    'model_name': 'answerdotai/ModernBERT-base',
    'max_len': 1024,
    'batch_size': 16,
    'epochs': 15, # Increased to allow early stopping to work
    'learning_rate': 2e-5,
    'gamma': 3.0, # Focal Loss Gamma
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'patience': 3 # Early Stopping Patience
}

# Hierarchy Definitions
EVASION_LABELS = [
    'Claims ignorance', 'Clarification', 'Declining to answer',
    'Deflection', 'Dodging', 'Explicit', 'General', 'Implicit', 'Partial/half-answer'
]
CLARITY_LABELS = ['Ambivalent', 'Clear Non-Reply', 'Clear Reply']
ALL_NODES = ['Root'] + CLARITY_LABELS + EVASION_LABELS
NODE2ID = {label: i for i, label in enumerate(ALL_NODES)}

# Hierarchy Parent -> Child Map
HIERARCHY = {
    'Root': ['Ambivalent', 'Clear Non-Reply', 'Clear Reply'],
    'Clear Reply': ['Explicit'],
    'Ambivalent': ['Implicit', 'General', 'Partial/half-answer', 'Dodging', 'Deflection'],
    'Clear Non-Reply': ['Declining to answer', 'Claims ignorance', 'Clarification']
}

# --- 3. Data Processing ---

def handle_dataset_upload():
    """Handles uploading and unzipping the dataset."""
    zip_name = 'processed_dataset.zip'
    if os.path.exists('train') and os.path.exists('test'):
        print("Dataset folders found. Skipping extraction.")
        return

    if not os.path.exists(zip_name):
        print(f"Please upload '{zip_name}' containing your train/test splits.")
        uploaded = files.upload()
        if zip_name not in uploaded:
             found_zip = [f for f in uploaded.keys() if f.endswith('.zip')]
             if found_zip:
                 zip_name = found_zip[0]
                 print(f"Using uploaded file: {zip_name}")
             else:
                 raise FileNotFoundError("Uploaded file is not a zip archive.")

    print(f"Extracting '{zip_name}'...")
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Extraction complete.")

class EvasionDataset(Dataset):
    def __init__(self, df, tokenizer, split='train'):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = CONFIG['max_len']
        self.split = split
        self.q_col = 'formatted_question' if 'formatted_question' in df.columns else 'interview_question'
        self.a_col = 'interview_answer'

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = f"{row[self.q_col]} {self.tokenizer.sep_token} {row[self.a_col]}"
        inputs = self.tokenizer(
            text, max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt"
        )
        if self.split == 'train':
            label_str = row['evasion_label']
            label_id = EVASION_LABELS.index(label_str) if label_str in EVASION_LABELS else -1
        else:
            label_id = -1
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(label_id, dtype=torch.long),
            'index': torch.tensor(idx, dtype=torch.long)
        }

# --- 4. Helper Classes (Loss, Model, EarlyStopping) ---

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean': return focal_loss.mean()
        elif self.reduction == 'sum': return focal_loss.sum()
        else: return focal_loss

class EarlyStopping:
    """Early stops the training if validation F1 doesn't improve after a given patience."""
    def __init__(self, patience=3, verbose=True, path='best_model.pth'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_score_max = -np.inf
        self.path = path

    def __call__(self, score, model):
        # We are maximizing F1, so higher is better
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(score, model)
        elif score <= self.best_score:
            self.counter += 1
            if self.verbose:
                print(f'⚠️ EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(score, model)
            self.counter = 0

    def save_checkpoint(self, score, model):
        if self.verbose:
            print(f'✅ Validation F1 improved ({self.val_score_max:.4f} --> {score:.4f}). Saving model...')
        torch.save(model.state_dict(), self.path)
        self.val_score_max = score

class GraphConvolution(nn.Module):
    def __init__(self, in_features, out_features, adj):
        super(GraphConvolution, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.adj = adj
        nn.init.xavier_uniform_(self.weight)

    def forward(self, input):
        support = torch.mm(input, self.weight)
        output = torch.mm(self.adj.to(input.device), support)
        return output

class HiAGM_ModernBERT(nn.Module):
    def __init__(self, num_nodes, adj_matrix, hidden_dim=768, gcn_dim=768):
        super(HiAGM_ModernBERT, self).__init__()
        self.bert = AutoModel.from_pretrained(CONFIG['model_name'])
        self.label_embedding = nn.Parameter(torch.FloatTensor(num_nodes, gcn_dim))
        nn.init.xavier_uniform_(self.label_embedding)
        self.gcn = GraphConvolution(gcn_dim, gcn_dim, adj_matrix)
        self.text_proj = nn.Linear(hidden_dim, gcn_dim)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = outputs.last_hidden_state[:, 0, :]
        text_rep = self.text_proj(cls_rep)
        label_reps = F.relu(self.gcn(self.label_embedding))
        logits = torch.mm(text_rep, label_reps.t())
        evasion_indices = [NODE2ID[l] for l in EVASION_LABELS]
        return logits[:, evasion_indices]

def build_adjacency_matrix():
    num_nodes = len(ALL_NODES)
    adj = np.eye(num_nodes)
    for parent, children in HIERARCHY.items():
        if parent not in NODE2ID: continue
        parent_idx = NODE2ID[parent]
        for child in children:
            if child not in NODE2ID: continue
            child_idx = NODE2ID[child]
            adj[parent_idx, child_idx] = 1
            adj[child_idx, parent_idx] = 1
    rowsum = np.array(adj.sum(1))
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    return torch.tensor(d_mat_inv_sqrt.dot(adj).dot(d_mat_inv_sqrt), dtype=torch.float32)

# --- 5. Evaluation Logic ---

def f1_for_class(gold_annotations, predictions, target_class):
    TP = FP = FN = 0
    for gold, pred in zip(gold_annotations, predictions):
        gold_set = set(gold)
        if pred == target_class and target_class in gold_set: TP += 1
        elif pred == target_class and target_class not in gold_set: FP += 1
        elif target_class in gold_set and pred not in gold_set: FN += 1
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

def evaluate_test_set(model, loader, test_df):
    model.eval()
    all_preds = []
    all_indices = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(CONFIG['device'])
            mask = batch['attention_mask'].to(CONFIG['device'])
            idxs = batch['index']
            logits = model(input_ids, mask)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_indices.extend(idxs.numpy())

    pred_strings = [EVASION_LABELS[p] for p in all_preds]
    idx_to_pred = {idx: pred for idx, pred in zip(all_indices, pred_strings)}

    aligned_preds = []
    aligned_golds = []

    for i, row in test_df.iterrows():
        if i not in idx_to_pred: continue
        aligned_preds.append(idx_to_pred[i])
        valid_labels = set()
        for col in ['annotator1', 'annotator2', 'annotator3']:
            if pd.notna(row.get(col)): valid_labels.add(row[col])
        aligned_golds.append(valid_labels)

    f1_scores = []
    print(f"\n{'Class':<25} | {'F1 Score':<10}")
    print("-" * 38)
    for label in EVASION_LABELS:
        score = f1_for_class(aligned_golds, aligned_preds, label)
        f1_scores.append(score)
        print(f"{label:<25} | {score:.4f}")

    macro_f1 = sum(f1_scores) / len(f1_scores)
    print("-" * 38)
    print(f"Macro F1 Score: {macro_f1:.4f}")
    return macro_f1

# --- 6. Main Execution ---

def main():
    handle_dataset_upload()

    try:
        print("Loading Dataset...")
        if os.path.exists('dataset_dict.json'):
            full_ds = load_from_disk(".")
            train_arrow = full_ds['train']
            test_arrow = full_ds['test']
        elif os.path.exists('train') and os.path.exists('test'):
             train_arrow = load_from_disk("train")
             test_arrow = load_from_disk("test")
        else:
             raise FileNotFoundError("Could not find dataset folders.")

        train_df = train_arrow.to_pandas()
        test_df = test_arrow.to_pandas()
        print(f"Loaded Train: {len(train_df)} rows, Test: {len(test_df)} rows")

        print("Computing Class Weights for Focal Loss...")
        y_train = [y for y in train_df['evasion_label'].tolist() if y in EVASION_LABELS]
        weights = compute_class_weight('balanced', classes=np.unique(EVASION_LABELS), y=y_train)
        weight_dict = dict(zip(np.unique(EVASION_LABELS), weights))
        ordered_weights = [weight_dict[l] for l in EVASION_LABELS]
        class_weights_tensor = torch.tensor(ordered_weights, dtype=torch.float32).to(CONFIG['device'])
        print(f"Class Weights: {class_weights_tensor}")

        tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
        train_dataset = EvasionDataset(train_df, tokenizer, split='train')
        test_dataset = EvasionDataset(test_df, tokenizer, split='test')

        train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

        adj = build_adjacency_matrix()
        model = HiAGM_ModernBERT(num_nodes=len(ALL_NODES), adj_matrix=adj)
        model.to(CONFIG['device'])

        optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'])
        total_steps = len(train_loader) * CONFIG['epochs']
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
        criterion = FocalLoss(alpha=class_weights_tensor, gamma=CONFIG['gamma'])

        # --- INITIALIZE EARLY STOPPING ---
        early_stopper = EarlyStopping(patience=CONFIG['patience'], path='best_model_hiagm.pth')

        print(f"Starting training on {CONFIG['device']} for {CONFIG['epochs']} epochs...")

        for epoch in range(CONFIG['epochs']):
            model.train()
            total_loss = 0
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}")

            for step, batch in enumerate(progress_bar):
                b_ids = batch['input_ids'].to(CONFIG['device'])
                b_mask = batch['attention_mask'].to(CONFIG['device'])
                b_labels = batch['labels'].to(CONFIG['device'])

                model.zero_grad()
                logits = model(b_ids, b_mask)
                loss = criterion(logits, b_labels)

                loss.backward()
                optimizer.step()
                scheduler.step()

                total_loss += loss.item()
                progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

            avg_loss = total_loss / len(train_loader)
            print(f"\nEnd of Epoch {epoch+1} | Average Train Loss: {avg_loss:.4f}")

            # Evaluate
            val_f1 = evaluate_test_set(model, test_loader, test_df)

            # --- EARLY STOPPING CHECK ---
            early_stopper(val_f1, model)

            if early_stopper.early_stop:
                print("⛔ Early stopping triggered. Stopping training.")
                break

        # Load best model
        print("Loading best model...")
        model.load_state_dict(torch.load('best_model_hiagm.pth'))
        print("Done.")

    except Exception as e:
        print(f"An error occurred: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()